In [1]:
import os, sys
ROOT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) in ("visualization", "notebooks", "scripts") else os.path.abspath(".")
if os.path.exists(os.path.join(ROOT_DIR, "GNN-jed")):
    os.chdir(os.path.join(ROOT_DIR, "GNN-jed"))
import os
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, knn_graph
from torch_geometric.loader import DataLoader
import numpy as np

## Making the data suitable for pytorch

In [2]:
def load_data_and_create_graphs(data_folder_path, labels_folder_path, output_folder_path, n, outputlist, K):
    os.makedirs(output_folder_path, exist_ok=True)
    for output in outputlist:
        data_path = f"{data_folder_path}obs_matrices_snapshot_{output:04d}.npy"
        label_path = f"{labels_folder_path}B_L_snapshot_{output:04d}.npy"

        data_matrix3d = np.load(data_path)
        label_matrix = np.load(label_path)
        
        for i in range(n):
            data_matrix = data_matrix3d[i][:,:5] # Use only the first 5 feature vithout the v_los
            
            edge_index = knn_graph(torch.tensor(data_matrix, dtype=torch.float), k=K, loop=False)
            graph_data = Data(x=torch.tensor(data_matrix, dtype=torch.float), edge_index=edge_index)
            label = torch.tensor(label_matrix, dtype=torch.float)
            
            torch.save(graph_data, f'{output_folder_path}/graph_data_output_{output}_rotation_{i}.pt')
            torch.save(label, f'{output_folder_path}/label_output_{output}_rotation_{i}.pt')


In [16]:

data_folder = 'obs_all_stars/'
labels_folder = 'labels_bound_stars/'
n = 5                                   # distinct rotations 
outputlist = list(range(0,101))         # 101 snapshots per rotation (n) per IC_
output_folder = '../../../../../../../../Volumes/Adrian 1 SS/MA2/graph_data/'
K = 30                                  # Number of nearest neighbors for graph construction

for dir in os.listdir(data_folder):
    data_folder_path = data_folder + dir + '/'
    labels_folder_path = labels_folder + dir + '/'
    output_folder_path = output_folder + dir + '/'
    if not os.path.exists(output_folder_path):
        print(dir)
        load_data_and_create_graphs(data_folder_path, labels_folder_path, output_folder_path, n, outputlist, K)

IC_18
IC_1
IC_8
IC_11


KeyboardInterrupt: 